# 🚀 TIKTOK VIDEO SCRAPER - GOOGLE DRIVE & COLAB INTEGRATED EDITION
### Công cụ cào video TikTok đa luồng kết nối & lưu trữ trực tiếp lên Google Drive

---

## 📂 BƯỚC 1: KẾT NỐI VỚI GOOGLE DRIVE & CÀI THƯ VIỆN
*Nhấp chạy ô này để kết nối Google Colab với tài khoản Google Drive của bạn.*

In [ ]:
# 1. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Cài đặt các thư viện cần thiết
!pip install -q ddgs duckduckgo-search pandas
print("✅ Đã kết nối Google Drive và cài đặt xong thư viện!")

## 📥 BƯỚC 2: KIỂM TRA / UPLOAD FILE DANH SÁCH (`danh_sach_phuong.csv`)
*Tool sẽ tự động tìm file `danh_sach_phuong.csv` trong Google Drive của bạn (tại `/content/drive/MyDrive/`). Nếu chưa có, ô bên dưới sẽ hỗ trợ upload lên.*

In [ ]:
import os
from google.colab import files

drive_input_path = '/content/drive/MyDrive/danh_sach_phuong.csv'
local_input_path = 'danh_sach_phuong.csv'

if os.path.exists(drive_input_path):
    print(f"✅ Đã tìm thấy file danh sách trên Google Drive tại: {drive_input_path}")
elif os.path.exists(local_input_path):
    print(f"✅ Đã tìm thấy file danh sách trên Colab tại: {local_input_path}")
else:
    print("📂 Không tìm thấy 'danh_sach_phuong.csv' trên Drive. Vui lòng chọn file từ máy tính để tải lên Drive:")
    uploaded = files.upload()
    # Tự động copy sang Google Drive để lưu lâu dài
    for fname in uploaded.keys():
        if fname.endswith('.csv'):
            os.system(f'cp "{fname}" "{drive_input_path}"')
            print(f"✅ Đã lưu file '{fname}' vào Google Drive ({drive_input_path})")

## ⚙️ BƯỚC 3: MÃ NGUỒN CÀO ĐA LUỒNG TỰ ĐỘNG LƯU DRIVE

In [ ]:
import csv
import os
import sys
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from ddgs import DDGS

csv_lock = threading.Lock()
print_lock = threading.Lock()

def safe_print(msg: str):
    with print_lock:
        print(msg)
        sys.stdout.flush()

def get_processed_locations(output_csv_path: str) -> set:
    processed = set()
    if os.path.exists(output_csv_path) and os.path.getsize(output_csv_path) > 0:
        try:
            with open(output_csv_path, mode='r', encoding='utf-8-sig') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    phuong = row.get('phuong', '').strip()
                    thanh_pho = row.get('thanh_pho', '').strip()
                    if phuong and thanh_pho:
                        processed.add((phuong, thanh_pho))
        except Exception as e:
            safe_print(f"[!] Lưu ý khi đọc file kết quả cũ: {e}")
    return processed

def safe_search_query_thread(query: str, max_results: int = 40, max_retries: int = 3):
    for attempt in range(1, max_retries + 1):
        try:
            with DDGS() as ddgs:
                results = list(ddgs.text(query, max_results=max_results))
                return results
        except Exception:
            if attempt < max_retries:
                time.sleep(attempt * 2)
            else:
                return []
    return []

def process_single_location(phuong: str, thanh_pho: str, target_count: int = 15):
    keyword = f"quán ăn ở {phuong} {thanh_pho}".strip()
    queries = [
        f'site:tiktok.com/@ "{keyword}"',
        f'site:tiktok.com/video "{keyword}"',
        f'tiktok video "{keyword}"',
        f'hashtag {keyword} site:tiktok.com',
        f'tiktok "{keyword}"',
        f'"{keyword}" site:tiktok.com'
    ]
    
    videos_data = []
    seen_ids = set()
    
    for q in queries:
        if len(videos_data) >= target_count:
            break
            
        results = safe_search_query_thread(q, max_results=40, max_retries=3)
        if not results:
            continue
            
        for r in results:
            url = r.get('href', '')
            title = r.get('title', '')
            snippet = r.get('body', '')
            
            if '/video/' in url:
                clean_url = url.split('?')[0]
                parts = clean_url.split('/')
                
                video_id = ''
                author = ''
                for idx, p in enumerate(parts):
                    if p.startswith('@'):
                        author = p.replace('@', '')
                    elif p == 'video' and idx + 1 < len(parts):
                        video_id = parts[idx + 1]
                        
                if video_id and video_id not in seen_ids:
                    seen_ids.add(video_id)
                    standard_url = f"https://www.tiktok.com/@{author}/video/{video_id}" if author else clean_url
                    
                    videos_data.append({
                        'phuong': phuong,
                        'thanh_pho': thanh_pho,
                        'search_keyword': keyword,
                        'id': video_id,
                        'author': author,
                        'webVideoUrl': standard_url,
                        'title': title,
                        'snippet': snippet
                    })
                    if len(videos_data) >= target_count:
                        break
                        
    return phuong, thanh_pho, videos_data

def run_colab_drive_scraper(input_filename="danh_sach_phuong.csv", output_filename="ket_qua_quan_an_phuong.csv", 
                            drive_folder="/content/drive/MyDrive", target_count=15, max_workers=5, 
                            start_from=None, limit=None):
    # Định vị đường dẫn file trên Google Drive
    input_path = os.path.join(drive_folder, input_filename) if os.path.exists(os.path.join(drive_folder, input_filename)) else input_filename
    output_path = os.path.join(drive_folder, output_filename) if os.path.exists(drive_folder) else output_filename

    safe_print(f"📂 File danh sách đầu vào: {input_path}")
    safe_print(f"💾 File kết quả đầu ra trên Google Drive: {output_path}")

    if not os.path.exists(input_path):
        safe_print(f"[!] Lỗi: Không tìm thấy file '{input_path}'. Vui lòng tải file vào Google Drive!")
        return

    processed_locations = get_processed_locations(output_path)
    if processed_locations:
        safe_print(f"[*] Đã quét xong {len(processed_locations)} địa điểm trên Drive. Sẽ bỏ qua.")

    locations = []
    with open(input_path, mode='r', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        for row in reader:
            phuong = row.get('Tên', '').strip()
            thanh_pho = row.get('Thành Phố', '').strip()
            if phuong and thanh_pho:
                locations.append((phuong, thanh_pho))

    if start_from:
        start_clean = start_from.strip().lower()
        start_idx = None
        for i, (p, tp) in enumerate(locations):
            full_str = f"{p},{tp}".lower()
            if start_clean in full_str or start_clean in f"{p} {tp}".lower() or start_clean in p.lower():
                start_idx = i
                safe_print(f"[*] Định vị điểm bắt đầu tại dòng {i+1}: '{p}, {tp}'")
                break
        if start_idx is not None:
            locations = locations[start_idx:]

    unprocessed = [loc for loc in locations if loc not in processed_locations]
    safe_print(f"[*] Số địa điểm cần quét còn lại: {len(unprocessed)}")

    if limit and limit > 0:
        unprocessed = unprocessed[:limit]

    if not unprocessed:
        safe_print("[+] Tất cả địa điểm được chọn đã được quét xong hoàn tất!")
        return

    file_exists = os.path.exists(output_path) and os.path.getsize(output_path) > 0
    fieldnames = ['phuong', 'thanh_pho', 'search_keyword', 'id', 'author', 'webVideoUrl', 'title', 'snippet']

    if not file_exists:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        with open(output_path, mode='a', newline='', encoding='utf-8-sig') as f_out:
            writer = csv.DictWriter(f_out, fieldnames=fieldnames)
            writer.writeheader()

    start_time = time.time()
    completed_count = 0
    total_to_process = len(unprocessed)
    
    safe_print(f"\n🚀 BẮT ĐẦU CHẠY ĐA LUỒNG - DỮ LIỆU GHI TRỰC TIẾP LÊN GOOGLE DRIVE...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_loc = {
            executor.submit(process_single_location, p, tp, target_count): (p, tp) 
            for (p, tp) in unprocessed
        }
        for future in as_completed(future_to_loc):
            p, tp = future_to_loc[future]
            try:
                res_phuong, res_tp, vids = future.result()
                completed_count += 1
                if vids:
                    with csv_lock:
                        with open(output_path, mode='a', newline='', encoding='utf-8-sig') as f_out:
                            writer = csv.DictWriter(f_out, fieldnames=fieldnames)
                            writer.writerows(vids)
                            f_out.flush()
                    safe_print(f"[{completed_count}/{total_to_process}]  {res_phuong}, {res_tp} -> Đã lưu {len(vids)}/{target_count} video lên Google Drive")
                else:
                    safe_print(f"[{completed_count}/{total_to_process}] ⚠️ {res_phuong}, {res_tp} -> 0 video")
            except Exception as exc:
                safe_print(f"[!] Lỗi khi xử lý {p}, {tp}: {exc}")

    elapsed = time.time() - start_time
    safe_print(f"\n[🎉] ĐÃ HOÀN THÀNH {completed_count} ĐỊA ĐIỂM TRONG {elapsed:.1f} GIÂY!")
    safe_print(f"📁 Dữ liệu lưu an toàn tại Google Drive: {output_path}")

## ▶️ BƯỚC 4: THIẾT LẬP THAM SỐ & KHỞI CHẠY TỰ ĐỘNG ĐỒNG BỘ DRIVE

In [ ]:
# === CẤU HÌNH THAM SỐ ĐỒNG BỘ GOOGLE DRIVE ==
DRIVE_FOLDER = "/content/drive/MyDrive" # Thư mục Google Drive lưu trữ dữ liệu
START_FROM = "Hà Đông"                  # Điểm bắt đầu quét (Ví dụ: "Hà Đông" hoặc None)
NUM_WORKERS = 5                         # Số luồng song song trên Colab (Khuyên dùng 5 - 10)
VIDEOS_PER_LOCATION = 15                # Số lượng video tối đa mỗi địa điểm
LIMIT_LOCATIONS = None                 # Giới hạn số lượng địa điểm (None để quét tất cả)

# Khởi chạy cào dữ liệu và ghi trực tiếp vào Google Drive
run_colab_drive_scraper(
    input_filename="danh_sach_phuong.csv",
    output_filename="ket_qua_quan_an_phuong.csv",
    drive_folder=DRIVE_FOLDER,
    target_count=VIDEOS_PER_LOCATION,
    max_workers=NUM_WORKERS,
    start_from=START_FROM,
    limit=LIMIT_LOCATIONS
)

## 📁 BƯỚC 5: TẢI TRỰC TIẾP TỪ COLAB VỀ MÁY TÍNH (NẾU CẦN)
*Lưu ý: Dữ liệu đã được lưu tự động trên Google Drive của bạn (`ket_qua_quan_an_phuong.csv`). Bạn có thể tải thêm 1 bản về máy tính tại đây.*

In [ ]:
from google.colab import files
import os

output_file = '/content/drive/MyDrive/ket_qua_quan_an_phuong.csv'
if os.path.exists(output_file):
    print(f"📥 Đang tải file kết quả từ Google Drive về máy tính của bạn...")
    files.download(output_file)
else:
    print(f"[!] Không tìm thấy file '{output_file}'!")